# MAPF Comparison — progress-visible version

This version is designed to avoid the impression that the notebook is frozen:
- prints a status message **before each algorithm**;
- runs a small POPS sample by default and reports progress for every order;
- uses a bounded space-time horizon and PBS high-level node limit;
- saves partial results after each algorithm, so completed measurements are not lost.

Keep `cooperative_astar.py`, `pops_cooperative_astar.py`, and `pbs_mapf.py` in the same folder as this notebook. Input CSVs must be in `results/`: `grid.csv`, `start.csv`, and `goal.csv`.

**If it still takes too long, stop the cell and reduce `POPS_ORDERS` or `PBS_NODE_LIMIT`.** A run that reaches a limit is not proof that no solution exists.


In [ ]:
import os
import time
import itertools
import numpy as np
import pandas as pd
from IPython.display import display

from cooperative_astar import prioritized_cooperative_astar
from pops_cooperative_astar import generate_priority_orders
from pbs_mapf import priority_based_search

DATA_FOLDER = "results"
POPS_ORDERS = 8        # Start small; increase after confirming the notebook runs
POPS_SEED = 42
PBS_NODE_LIMIT = 100   # High-level PBS expansions
MAX_TIME = None        # Set an integer such as 150 to impose a time horizon
OUTPUT_CSV = os.path.join(DATA_FOLDER, "comparison_results.csv")

os.makedirs(DATA_FOLDER, exist_ok=True)
print("Imports complete.", flush=True)


In [ ]:
def load_inputs(folder=DATA_FOLDER):
    required = ["grid.csv", "start.csv", "goal.csv"]
    missing = [name for name in required if not os.path.isfile(os.path.join(folder, name))]
    if missing:
        raise FileNotFoundError(
            f"Missing {missing} in {folder!r}. Put the three CSV files in that folder."
        )
    grid = np.loadtxt(os.path.join(folder, "grid.csv"), delimiter=",", dtype=int)
    grid = np.atleast_2d(grid)
    starts_arr = np.atleast_2d(np.loadtxt(os.path.join(folder, "start.csv"), delimiter=",", dtype=int))
    goals_arr = np.atleast_2d(np.loadtxt(os.path.join(folder, "goal.csv"), delimiter=",", dtype=int))
    starts = [tuple(map(int, row)) for row in starts_arr]
    goals = [tuple(map(int, row)) for row in goals_arr]
    if len(starts) != len(goals):
        raise ValueError("Number of starts and goals differ.")
    if len(set(starts)) != len(starts) or len(set(goals)) != len(goals):
        raise ValueError("Starts and goals should be unique.")
    if any(grid[s] == 1 for s in starts) or any(grid[g] == 1 for g in goals):
        raise ValueError("A start or goal is on an obstacle.")
    return grid, starts, goals

grid, starts, goals = load_inputs()
n = len(starts)
print(f"Loaded {grid.shape[0]}x{grid.shape[1]} grid, {n} robots, {int(np.sum(grid == 1))} obstacles.", flush=True)
print("Starts:", starts)
print("Goals: ", goals)
if MAX_TIME is None:
    MAX_TIME = max(2 * grid.size, 50)
print(f"Search time horizon: {MAX_TIME}", flush=True)


## Run the comparison

Makespan and total path length are reported only when **all robots succeed**. POPS tests only a bounded sample of orders in this notebook; it does not exhaustively search all 12! orders.


In [ ]:
def summarise(name, results, n, elapsed, notes=""):
    if results is None:
        return {
            "Algorithm": name, "Solved robots": f"0/{n}", "Success rate (%)": 0.0,
            "Makespan": np.nan, "Total path length": np.nan,
            "Nodes expanded": np.nan, "Runtime (s)": round(elapsed, 4), "Notes": notes
        }
    robots = [results.get(f"Robot {i+1}", {}) for i in range(n)]
    count = sum(bool(r.get("success")) for r in robots)
    complete = count == n
    arrivals = [r.get("arrival_time") for r in robots if r.get("success") and r.get("arrival_time") is not None]
    lengths = [r.get("path_length") for r in robots if r.get("success") and r.get("path_length") is not None]
    nodes = sum(r.get("nodes_expanded", 0) for r in robots)
    return {
        "Algorithm": name, "Solved robots": f"{count}/{n}",
        "Success rate (%)": round(100 * count / n, 2),
        "Makespan": max(arrivals) if complete and arrivals else np.nan,
        "Total path length": sum(lengths) if complete else np.nan,
        "Nodes expanded": nodes, "Runtime (s)": round(elapsed, 4), "Notes": notes
    }

rows = []

# 1. Fixed-priority Cooperative A*
print("\n[1/3] Running fixed-priority Cooperative A*...", flush=True)
t0 = time.perf_counter()
coop_results = prioritized_cooperative_astar(grid, starts, goals, max_time=MAX_TIME)
coop_seconds = time.perf_counter() - t0
rows.append(summarise("Fixed-priority Cooperative A*", coop_results, n, coop_seconds,
                      "Input order is used as priority"))
pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
print(f"Cooperative A* finished in {coop_seconds:.2f}s.", flush=True)
display(pd.DataFrame(rows))

# 2. POPS with visible per-order progress
print(f"\n[2/3] Running POPS on up to {POPS_ORDERS} orders...", flush=True)
orders = generate_priority_orders(n, max_orders=POPS_ORDERS, seed=POPS_SEED)
pops_best = None
pops_best_key = None
pops_best_order = None
pops_complete_orders = 0
pops_start_time = time.perf_counter()

for k, order in enumerate(orders, start=1):
    print(f"  POPS order {k}/{len(orders)}: {[i+1 for i in order]}", flush=True)
    trial = prioritized_cooperative_astar(
        grid, starts, goals, max_time=MAX_TIME, priority_order=order
    )
    success_count = sum(bool(trial[f"Robot {i+1}"].get("success")) for i in range(n))
    all_success = success_count == n
    arrivals = [trial[f"Robot {i+1}"].get("arrival_time") for i in range(n)
                if trial[f"Robot {i+1}"].get("success")]
    lengths = [trial[f"Robot {i+1}"].get("path_length") for i in range(n)
               if trial[f"Robot {i+1}"].get("success")]
    makespan = max(arrivals) if all_success and arrivals else float("inf")
    total_length = sum(lengths) if all_success else float("inf")
    key = (0 if all_success else 1, 0 if all_success else -success_count,
           makespan, total_length)
    if pops_best_key is None or key < pops_best_key:
        pops_best_key, pops_best, pops_best_order = key, trial, [i+1 for i in order]
    if all_success:
        pops_complete_orders += 1

pops_seconds = time.perf_counter() - pops_start_time
rows.append(summarise("POPS", pops_best, n, pops_seconds,
                      f"Orders tested: {len(orders)}; complete orders: {pops_complete_orders}; best order: {pops_best_order}"))
pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
print(f"POPS finished in {pops_seconds:.2f}s.", flush=True)
display(pd.DataFrame(rows))

# 3. PBS
print(f"\n[3/3] Running PBS (high-level node limit={PBS_NODE_LIMIT})...", flush=True)
t0 = time.perf_counter()
pbs_output = priority_based_search(
    grid, starts, goals, max_time=MAX_TIME,
    max_high_level_nodes=PBS_NODE_LIMIT
)
pbs_seconds = time.perf_counter() - t0
rows.append(summarise("PBS", pbs_output.get("results"), n, pbs_seconds,
                      f"High-level nodes: {pbs_output.get('high_level_nodes')}; {pbs_output.get('reason')}"))
pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
print(f"PBS finished in {pbs_seconds:.2f}s. Status: {pbs_output.get('reason')}", flush=True)

comparison = pd.DataFrame(rows)
display(comparison)
print(f"\nSaved results to {OUTPUT_CSV}", flush=True)


## Troubleshooting

- If no status appears after starting a cell, confirm the cell is actually running and that the kernel is responsive.
- If it stalls during **Cooperative A***, the issue is one space-time search or an overly large time horizon. Lower `MAX_TIME` (for example, to `100`) and retry.
- If it stalls during **POPS**, lower `POPS_ORDERS` to `2` or `3`. This version prints every order, so you can see where time is spent.
- If it stalls during **PBS**, lower `PBS_NODE_LIMIT` to `20` for a quick diagnostic.
- A smaller horizon/node limit makes runs faster but can cause false failures on difficult instances. Record the limits in your report.
- Restart the kernel and run cells from top to bottom after changing settings.
